## 1. Setup and Configuration

In [ ]:
# Import required libraries
from databricks import sql
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from matplotlib_venn import venn2, venn3, venn2_circles, venn3_circles
import seaborn as sns

In [ ]:
# Load environment variables from parent directory
load_dotenv(dotenv_path='../.env')

# Validate credentials
required_vars = ['DATABRICKS_SERVER_HOSTNAME', 'DATABRICKS_HTTP_PATH', 'DATABRICKS_TOKEN']
missing = [var for var in required_vars if not os.getenv(var)]
if missing:
    raise ValueError(f"Missing environment variables: {', '.join(missing)}")

print("✓ Environment configured")

## 2. Analysis Parameters

In [ ]:
# Customer and time period settings
customer_filter = 'cds_8007'  # SAPPORO DRUG JP
start_date = '2024-01-01'
end_date = '2024-03-31'

# Product filters
category_filter = 'Laundry'

# Brands to compare (2 or 3 brands for Venn diagram)
# For 2-way Venn:
brand_list = ['アリエール', 'ボールド']  # Must be 2 or 3 brands

# For 3-way Venn (uncomment and modify):
# brand_list = ['アリエール', 'ボールド', 'アタック']

venn_granularity = 'jp_sub_brand_alter_lang_name'  # Brand level to compare

if len(brand_list) not in [2, 3]:
    raise ValueError("brand_list must contain exactly 2 or 3 brands for Venn diagram")

print(f"✓ Parameters set")
print(f"  Analysis period: {start_date} to {end_date}")
print(f"  Category: {category_filter}")
print(f"  Comparing brands: {', '.join(brand_list)}")
print(f"  Venn diagram type: {len(brand_list)}-way")

## 3. Build and Execute Query

In [ ]:
# Build brand overlap query
brand_conditions = [f"'{brand}'" for brand in brand_list]
brands_in_clause = ', '.join(brand_conditions)

query = f"""
WITH base_transactions AS (
    SELECT
        idpos.shopper_key AS shopper_id,
        {venn_granularity} AS brand,
        pos_unit_sales_qty AS unit,
        pos_sales_amt AS value
    FROM
        cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
        LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
        LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE
        jp_category_name = '{category_filter}'
        AND idpos.data_provider_code_part = '{customer_filter}'
        AND sales_period_group_end_date_part BETWEEN '{start_date}' AND '{end_date}'
        AND shopper.member_ind = 'Y'
        AND {venn_granularity} IN ({brands_in_clause})
),
shopper_brands AS (
    SELECT
        shopper_id,
        brand,
        SUM(value) AS total_value,
        SUM(unit) AS total_unit,
        COUNT(*) AS purchase_count
    FROM base_transactions
    GROUP BY shopper_id, brand
)
SELECT
    shopper_id,
    COLLECT_SET(brand) AS brands_purchased,
    SUM(total_value) AS total_value,
    SUM(total_unit) AS total_unit
FROM shopper_brands
GROUP BY shopper_id
"""

print("✓ SQL query constructed")
print(f"  Query length: {len(query)} characters")

In [ ]:
# Execute query
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_SERVER_HOSTNAME"),
    http_path=os.getenv("DATABRICKS_HTTP_PATH"),
    access_token=os.getenv("DATABRICKS_TOKEN")
) as connection:
    with connection.cursor() as cursor:
        cursor.execute(query)
        result = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description]
        df = pd.DataFrame(result, columns=columns)

# Convert data types
df['total_value'] = pd.to_numeric(df['total_value'], errors='coerce')
df['total_unit'] = pd.to_numeric(df['total_unit'], errors='coerce')

print(f"✓ Query executed successfully")
print(f"  Retrieved {len(df)} unique shoppers")
print(f"\nFirst few rows:")
df.head(10)

## 4. Process and Analyze Results

In [ ]:
# Create brand membership flags
for brand in brand_list:
    df[f'has_{brand}'] = df['brands_purchased'].apply(lambda x: brand in x if x else False)

# Calculate overlap segments
if len(brand_list) == 2:
    # 2-way Venn
    brand_a, brand_b = brand_list
    
    only_a = df[df[f'has_{brand_a}'] & ~df[f'has_{brand_b}']]
    only_b = df[~df[f'has_{brand_a}'] & df[f'has_{brand_b}']]
    both = df[df[f'has_{brand_a}'] & df[f'has_{brand_b}']]
    
    segments = {
        f'Only {brand_a}': len(only_a),
        f'Only {brand_b}': len(only_b),
        f'Both': len(both)
    }
    
    total_a = len(only_a) + len(both)
    total_b = len(only_b) + len(both)
    overlap_rate = (len(both) / min(total_a, total_b) * 100) if min(total_a, total_b) > 0 else 0
    
elif len(brand_list) == 3:
    # 3-way Venn
    brand_a, brand_b, brand_c = brand_list
    
    only_a = df[df[f'has_{brand_a}'] & ~df[f'has_{brand_b}'] & ~df[f'has_{brand_c}']]
    only_b = df[~df[f'has_{brand_a}'] & df[f'has_{brand_b}'] & ~df[f'has_{brand_c}']]
    only_c = df[~df[f'has_{brand_a}'] & ~df[f'has_{brand_b}'] & df[f'has_{brand_c}']]
    ab = df[df[f'has_{brand_a}'] & df[f'has_{brand_b}'] & ~df[f'has_{brand_c}']]
    ac = df[df[f'has_{brand_a}'] & ~df[f'has_{brand_b}'] & df[f'has_{brand_c}']]
    bc = df[~df[f'has_{brand_a}'] & df[f'has_{brand_b}'] & df[f'has_{brand_c}']]
    abc = df[df[f'has_{brand_a}'] & df[f'has_{brand_b}'] & df[f'has_{brand_c}']]
    
    segments = {
        f'Only {brand_a}': len(only_a),
        f'Only {brand_b}': len(only_b),
        f'Only {brand_c}': len(only_c),
        f'{brand_a} & {brand_b}': len(ab),
        f'{brand_a} & {brand_c}': len(ac),
        f'{brand_b} & {brand_c}': len(bc),
        f'All Three': len(abc)
    }
    
    total_a = len(only_a) + len(ab) + len(ac) + len(abc)
    total_b = len(only_b) + len(ab) + len(bc) + len(abc)
    total_c = len(only_c) + len(ac) + len(bc) + len(abc)
    overlap_rate = ((len(ab) + len(ac) + len(bc) + len(abc)) / len(df) * 100) if len(df) > 0 else 0

total_shoppers = len(df)

print("=" * 60)
print("BRAND OVERLAP ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nTotal Shoppers: {total_shoppers:,}")
print(f"\nSegment Breakdown:")
for segment, count in segments.items():
    pct = (count / total_shoppers * 100) if total_shoppers > 0 else 0
    print(f"  {segment:30s}: {count:6,} ({pct:5.1f}%)")

if len(brand_list) == 2:
    print(f"\nBrand Totals:")
    print(f"  {brand_a:30s}: {total_a:6,}")
    print(f"  {brand_b:30s}: {total_b:6,}")
    print(f"  Overlap rate: {overlap_rate:.1f}%")
elif len(brand_list) == 3:
    print(f"\nBrand Totals:")
    print(f"  {brand_a:30s}: {total_a:6,}")
    print(f"  {brand_b:30s}: {total_b:6,}")
    print(f"  {brand_c:30s}: {total_c:6,}")
    print(f"  Any overlap rate: {overlap_rate:.1f}%")

## 5. Visualizations

In [ ]:
# Venn Diagram using matplotlib-venn
plt.figure(figsize=(12, 8))

if len(brand_list) == 2:
    # 2-way Venn
    venn = venn2(
        subsets=(len(only_a), len(only_b), len(both)),
        set_labels=(brand_a, brand_b),
        set_colors=('#FF6B6B', '#4ECDC4'),
        alpha=0.6
    )
    
    # Add circles
    venn2_circles(subsets=(len(only_a), len(only_b), len(both)), linewidth=2)
    
    # Customize labels
    for text in venn.set_labels:
        text.set_fontsize(14)
        text.set_fontweight('bold')
    
    for text in venn.subset_labels:
        if text:
            text.set_fontsize(12)
    
elif len(brand_list) == 3:
    # 3-way Venn
    venn = venn3(
        subsets=(len(only_a), len(only_b), len(ab), len(only_c), len(ac), len(bc), len(abc)),
        set_labels=(brand_a, brand_b, brand_c),
        set_colors=('#FF6B6B', '#4ECDC4', '#95E1D3'),
        alpha=0.6
    )
    
    # Add circles
    venn3_circles(subsets=(len(only_a), len(only_b), len(ab), len(only_c), len(ac), len(bc), len(abc)), linewidth=2)
    
    # Customize labels
    for text in venn.set_labels:
        text.set_fontsize(14)
        text.set_fontweight('bold')
    
    for text in venn.subset_labels:
        if text:
            text.set_fontsize(12)

plt.title(f'Brand Overlap Analysis: {category_filter} Category\n{start_date} to {end_date}', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Segment breakdown bar chart
segment_df = pd.DataFrame(list(segments.items()), columns=['Segment', 'Shoppers'])
segment_df['Percentage'] = (segment_df['Shoppers'] / total_shoppers * 100)
segment_df = segment_df.sort_values('Shoppers', ascending=True)

fig = go.Figure()

# Determine colors based on segment type
colors = []
for seg in segment_df['Segment']:
    if 'Only' in seg:
        colors.append('#4A90E2')  # Blue for exclusive
    elif 'All' in seg or 'Both' in seg:
        colors.append('#FF6B6B')  # Red for complete overlap
    else:
        colors.append('#FFA500')  # Orange for partial overlap

fig.add_trace(go.Bar(
    y=segment_df['Segment'],
    x=segment_df['Shoppers'],
    orientation='h',
    text=[f"{s:,} ({p:.1f}%)" for s, p in zip(segment_df['Shoppers'], segment_df['Percentage'])],
    textposition='outside',
    marker_color=colors
))

fig.update_layout(
    title='Shopper Distribution by Brand Overlap Segment',
    xaxis_title='Number of Shoppers',
    yaxis_title='Segment',
    height=500,
    showlegend=False
)

fig.show()

In [ ]:
# Exclusive vs Shared breakdown (pie chart)
if len(brand_list) == 2:
    exclusive_count = len(only_a) + len(only_b)
    shared_count = len(both)
    
    pie_data = pd.DataFrame({
        'Category': ['Exclusive to One Brand', 'Shared Between Brands'],
        'Shoppers': [exclusive_count, shared_count]
    })
    
elif len(brand_list) == 3:
    exclusive_count = len(only_a) + len(only_b) + len(only_c)
    partial_overlap = len(ab) + len(ac) + len(bc)
    full_overlap = len(abc)
    
    pie_data = pd.DataFrame({
        'Category': ['Exclusive to One Brand', '2-Brand Overlap', 'All 3 Brands'],
        'Shoppers': [exclusive_count, partial_overlap, full_overlap]
    })

fig = go.Figure(data=[go.Pie(
    labels=pie_data['Category'],
    values=pie_data['Shoppers'],
    hole=0.4,
    marker_colors=['#4A90E2', '#FFA500', '#FF6B6B'] if len(brand_list) == 3 else ['#4A90E2', '#FF6B6B'],
    texttemplate='%{label}<br>%{value:,}<br>(%{percent})',
    textfont_size=12
)])

fig.update_layout(
    title='Exclusive vs Shared Shoppers',
    height=500
)

fig.show()

In [ ]:
# Value contribution by segment
segment_values = {}

if len(brand_list) == 2:
    segment_values[f'Only {brand_a}'] = only_a['total_value'].sum()
    segment_values[f'Only {brand_b}'] = only_b['total_value'].sum()
    segment_values['Both'] = both['total_value'].sum()
    
elif len(brand_list) == 3:
    segment_values[f'Only {brand_a}'] = only_a['total_value'].sum()
    segment_values[f'Only {brand_b}'] = only_b['total_value'].sum()
    segment_values[f'Only {brand_c}'] = only_c['total_value'].sum()
    segment_values[f'{brand_a} & {brand_b}'] = ab['total_value'].sum()
    segment_values[f'{brand_a} & {brand_c}'] = ac['total_value'].sum()
    segment_values[f'{brand_b} & {brand_c}'] = bc['total_value'].sum()
    segment_values['All Three'] = abc['total_value'].sum()

value_df = pd.DataFrame(list(segment_values.items()), columns=['Segment', 'Total_Value'])
value_df = value_df.sort_values('Total_Value', ascending=True)

fig = go.Figure()

fig.add_trace(go.Bar(
    y=value_df['Segment'],
    x=value_df['Total_Value'],
    orientation='h',
    text=[f"¥{v:,.0f}" for v in value_df['Total_Value']],
    textposition='outside',
    marker_color='#32CD32'
))

fig.update_layout(
    title='Sales Value by Overlap Segment',
    xaxis_title='Total Sales Value (¥)',
    yaxis_title='Segment',
    height=500,
    showlegend=False
)

fig.show()

## 6. Data Tables

In [ ]:
# Segment summary table
summary_table = pd.DataFrame({
    'Segment': list(segments.keys()),
    'Shoppers': list(segments.values()),
    'Value': [segment_values[k] for k in segments.keys()]
})
summary_table['% of Shoppers'] = (summary_table['Shoppers'] / total_shoppers * 100).round(1)
summary_table['Avg Value per Shopper'] = (summary_table['Value'] / summary_table['Shoppers']).round(0)

print("Segment Summary:")
summary_table

## 7. Data Export

In [ ]:
# Export to Excel (optional)
export_file = f"venn_analysis_{category_filter}_{'_'.join(brand_list)}_{start_date}_to_{end_date}.xlsx"

with pd.ExcelWriter(export_file, engine='openpyxl') as writer:
    # Summary sheet
    summary_df = pd.DataFrame({
        'Metric': ['Total Shoppers', 'Brands Compared', 'Overlap Rate %'],
        'Value': [f"{total_shoppers:,}", ', '.join(brand_list), f"{overlap_rate:.1f}%"]
    })
    summary_df.to_excel(writer, sheet_name='Summary', index=False)
    
    # Segment breakdown
    summary_table.to_excel(writer, sheet_name='Segment_Breakdown', index=False)
    
    # Shopper details by segment
    if len(brand_list) == 2:
        only_a[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name=f'Only_{brand_a}', index=False)
        only_b[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name=f'Only_{brand_b}', index=False)
        both[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name='Both_Brands', index=False)
    elif len(brand_list) == 3:
        only_a[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name=f'Only_{brand_a}', index=False)
        only_b[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name=f'Only_{brand_b}', index=False)
        only_c[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name=f'Only_{brand_c}', index=False)
        abc[['shopper_id', 'total_value', 'total_unit']].to_excel(writer, sheet_name='All_Three_Brands', index=False)

print(f"✓ Data exported to: {export_file}")